In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

import huggingface_hub
from huggingface_hub import constants

print("Hub version:", huggingface_hub.__version__)
print("Environment variable:", os.getenv("HF_HUB_DISABLE_XET"))
print("Constant:", constants.HF_HUB_DISABLE_XET)

Hub version: 2.4.0
Environment variable: 1
Constant: True


In [2]:
from huggingface_hub import login
login()

In [3]:
# Hugging Face download configuration
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "3600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"

import time, torch, open_clip
device = "cuda" if torch.cuda.is_available() else "cpu"
SOURCES = {"OpenCLIP": ("ViT-B-16", "laion2b_s34b_b88k"),
           "BioCLIP": ("hf-hub:imageomics/bioclip", None)}

clip_models = {}
for name, (arch, weights) in SOURCES.items():
    start = time.perf_counter()
    model, _, preprocess = open_clip.create_model_and_transforms(arch, pretrained=weights)
    tokenizer = open_clip.get_tokenizer(arch)
    clip_models[name] = {"model": model.to(device).eval(), "preprocess": preprocess, "tokenizer": tokenizer}
    v = model.visual
    side = v.image_size[0] if isinstance(v.image_size, tuple) else v.image_size
    patch = v.patch_size[0] if isinstance(v.patch_size, tuple) else v.patch_size
    grid = side // patch
    print(f"{name}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters "
          f"(image {sum(p.numel() for p in v.parameters()) / 1e6:.1f} M, text "
          f"{sum(p.numel() for p in model.transformer.parameters()) / 1e6:.1f} M) | loaded in {time.perf_counter() - start:.0f} s")
    print(f"  image {side} x {side}, patches {patch} x {patch} -> {grid} x {grid} = {grid * grid} patch tokens "
          f"+ 1 class token = {grid * grid + 1} tokens | width {v.transformer.width}, "
          f"{v.transformer.layers} layers, {v.transformer.resblocks[0].attn.num_heads} heads | "
          f"embedding {model.text_projection.shape[1]}")
print(clip_models["BioCLIP"]["preprocess"])

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/laion/CLIP-ViT-B-16-laion2B-s34B-b88K/resolve/main/open_clip_model.safetensors: The read operation timed out
Trying to resume download...
Trying to resume download...
Error while downloading from https://huggingface.co/laion/CLIP-ViT-B-16-laion2B-s34B-b88K/resolve/main/open_clip_model.safetensors: The read operation timed out
Trying to resume download...
Trying to resume download...
Error while downloading from https://huggingface.co/laion/CLIP-ViT-B-16-laion2B-s34B-b88K/resolve/main/open_clip_model.safetensors: The read operation timed out
Trying to resume download...
Trying to resume download...
c:\Users\Jana\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jana\.cache\huggingface\hub\models--laion--CLIP-ViT-B-16-laion2B-s34B-b88K.

OpenCLIP: 149.6 M parameters (image 86.2 M, text 37.8 M) | loaded in 453 s
  image 224 x 224, patches 16 x 16 -> 14 x 14 = 196 patch tokens + 1 class token = 197 tokens | width 768, 12 layers, 12 heads | embedding 512


open_clip_pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

c:\Users\Jana\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jana\.cache\huggingface\hub\models--imageomics--bioclip. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


BioCLIP: 149.6 M parameters (image 86.2 M, text 37.8 M) | loaded in 769 s
  image 224 x 224, patches 16 x 16 -> 14 x 14 = 196 patch tokens + 1 class token = 197 tokens | width 768, 12 layers, 12 heads | embedding 512
Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeConvertMode()
    MaybeToTensor()
    Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
)


In [4]:
import json
import pandas as pd
from pathlib import Path
from plantdata import load_classes, read_manifest
from training import scores
from zeroshot import TEMPLATE_SETS, prompt, text_features, image_features, zero_shot_logits

class_ids, crops, crop_mask = load_classes("splits_v2")
class_crops = [crops[int(crop_mask[:, k].nonzero())] for k in range(len(class_ids))]
CACHE_DIR = Path("cache_256")
manifest = read_manifest(CACHE_DIR)
val = manifest[manifest.split == "val_wild"].reset_index(drop=True)
y_true = torch.tensor(val.class_index.to_numpy())
crop_idx = torch.tensor([crops.index(c) for c in val.crop])

print("example prompts for tomato_late_blight:")
for t in ["label", "photo", "leaf", "symptoms", "scientific"]:
    print(f"  {t:10s} {prompt('tomato_late_blight', 'tomato', t)}")

Path("features").mkdir(exist_ok=True)
rows = []
for name, clip in clip_models.items():
    start = time.perf_counter()
    img = image_features(clip, [CACHE_DIR / p for p in val.cache_path], device)
    torch.save(img, f"features/clip_{name}_val_wild.pt")
    print(f"{name}: {len(img)} val_wild photos embedded in {time.perf_counter() - start:.0f} s")
    scale = clip["model"].logit_scale.detach().exp().item()   # CLIP's learned temperature (about 100)
    for template_set in TEMPLATE_SETS:
        txt = text_features(clip, class_ids, class_crops, template_set, device).cpu()
        pred = zero_shot_logits(img, txt, crop_idx, crop_mask, scale).argmax(dim=1)
        s = scores(y_true.numpy(), pred.numpy(), class_ids)
        rows.append({"model": name, "prompts": template_set, "val_wild F1-16": round(s["macro_f1_16_disease"], 3),
                     "F1-20": round(s["macro_f1_20"], 3), "LB recall": round(s["late_blight_recall"], 3),
                     "accuracy": round(s["accuracy"], 3)})
selection = pd.DataFrame(rows)
selection.to_csv("results/q6_prompt_selection_val_wild.csv", index=False)
print(selection.to_string(index=False))

best = selection.loc[selection.groupby("model")["val_wild F1-16"].idxmax()]
chosen = {r.model: r.prompts for r in best.itertuples()}
Path("checkpoints/q6_prompts.json").write_text(json.dumps(
    {"chosen": chosen, "selected_on": "val_wild", "metric": "macro_f1_16_disease"}, indent=2))
print("chosen (val_wild only):", chosen)

example prompts for tomato_late_blight:
  label      tomato late blight
  photo      a photo of tomato late blight.
  leaf       a photo of a tomato leaf with late blight.
  symptoms   a close-up photo of a tomato leaf showing symptoms of late blight, a plant disease.
  scientific a photo of a leaf of tomato (Solanum lycopersicum) with late blight.
OpenCLIP: 1630 val_wild photos embedded in 22 s
BioCLIP: 1630 val_wild photos embedded in 8 s
   model    prompts  val_wild F1-16  F1-20  LB recall  accuracy
OpenCLIP      label           0.341  0.312      0.251     0.396
OpenCLIP      photo           0.366  0.437      0.511     0.456
OpenCLIP       leaf           0.413  0.492      0.650     0.512
OpenCLIP   symptoms           0.386  0.440      0.306     0.428
OpenCLIP scientific           0.400  0.473      0.791     0.508
OpenCLIP   ensemble           0.416  0.492      0.703     0.499
 BioCLIP      label           0.355  0.379      0.076     0.412
 BioCLIP      photo           0.382  0.394 

In [5]:
import numpy as np
from PIL import Image

chosen = json.loads(Path("checkpoints/q6_prompts.json").read_text())["chosen"]
lab = manifest[manifest.split == "val_lab"].reset_index(drop=True)
lab_crop_idx = torch.tensor([crops.index(c) for c in lab.crop])
val_paths = [CACHE_DIR / p for p in val.cache_path]
sample = list(range(0, len(val), max(1, len(val) // 100)))[:100]      # the same 100 photos as the ANN latency

def time_per_image(predict_one, warmup=5):
    for i in sample[:warmup]:
        predict_one(i)
    times = []
    for i in sample:
        t0 = time.perf_counter()
        predict_one(i)
        times.append(time.perf_counter() - t0)
    return 1000 * float(np.median(times))

rows = []
for name, clip in clip_models.items():
    model = clip["model"]
    txt = text_features(clip, class_ids, class_crops, chosen[name], device).cpu()   # once, at start-up
    scale = model.logit_scale.detach().exp().item()
    img_lab = image_features(clip, [CACHE_DIR / p for p in lab.cache_path], device)
    torch.save(img_lab, f"features/clip_{name}_val_lab.pt")
    s_lab = scores(lab.class_index.to_numpy(), zero_shot_logits(img_lab, txt, lab_crop_idx, crop_mask, scale)
                   .argmax(dim=1).numpy(), class_ids)
    img_wild = torch.load(f"features/clip_{name}_val_wild.pt")
    s_wild = scores(y_true.numpy(), zero_shot_logits(img_wild, txt, crop_idx, crop_mask, scale)
                    .argmax(dim=1).numpy(), class_ids)
    row = {"model": f"{name} zero-shot ({chosen[name]})", "labelled training photos": 0,
           "parameters (M)": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
           "val_lab F1-16": round(s_lab["macro_f1_16_disease"], 3),
           "val_wild F1-16": round(s_wild["macro_f1_16_disease"], 3), "val_wild F1-20": round(s_wild["macro_f1_20"], 3),
           "LB recall": round(s_wild["late_blight_recall"], 3)}
    for dev in (["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"]):
        model.to(dev)
        txt_dev = txt.to(dev)
        @torch.no_grad()
        def clip_one(i):
            x = clip["preprocess"](Image.open(val_paths[i]).convert("RGB"))[None].to(dev)
            emb = model.encode_image(x)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            logits = zero_shot_logits(emb, txt_dev, crop_idx[i:i + 1].to(dev), crop_mask.to(dev), scale)
            return logits.argmax(1).item()
        row[f"ms per image {dev.upper()}"] = round(time_per_image(clip_one), 1)
    model.to(device)
    rows.append(row)

clip_table = pd.DataFrame(rows)
clip_table.to_csv("results/q6_clip_validation.csv", index=False)
print(clip_table.to_string(index=False))

                        model  labelled training photos  parameters (M)  val_lab F1-16  val_wild F1-16  val_wild F1-20  LB recall  ms per image CUDA  ms per image CPU
OpenCLIP zero-shot (ensemble)                         0           149.6          0.339           0.416           0.492      0.703               18.3             126.4
 BioCLIP zero-shot (ensemble)                         0           149.6          0.368           0.428           0.439      0.479               12.5             114.6


In [6]:
import numpy as np
from ood import raw_logits, familiarity_scores, separation, PathDataset
from plantdata import PlantDataset, eval_transform, make_loader
from plantmodel import CropMaskedResNet
from zeroshot import CROP_TEXT

# the same in-scope and out-of-scope photos as the ANN's warning (cell O1)
excluded = [c["id"] for c in json.loads(Path("class_mapping.json").read_text())["classes"] if c["tier"] == "candidate"]
index = pd.read_csv("audit_outputs/image_index.csv")
ood_rows = index[(index.dataset == "plantdoc") & index.class_id.isin(excluded)].reset_index(drop=True)
n = len(ood_rows)
ood_crop = torch.arange(len(crops)).repeat_interleave(n)          # every out-of-scope photo under every crop
photo_crop = np.tile(ood_rows.class_id.str.split("_").str[0].to_numpy(), len(crops))
same_crop = photo_crop == np.array(crops)[ood_crop.numpy()]          # e.g. a tomato pest photo with "tomato" chosen
print(f"out-of-scope: {n} photos x {len(crops)} crops = {n * len(crops)} (photo, crop) pairs; "
      f"{same_crop.sum()} pairs are the right crop with a problem the model does not know")

# 1. the ANN's warning (R15 energy), recomputed so it can be split by kind
ck = torch.load("checkpoints/R15.pt", map_location="cpu", weights_only=False)
ann = CropMaskedResNet(ck["config"]["depth"], crop_mask, pretrained=False)
ann.load_state_dict(ck["state_dict"])
ann.to(device).eval()
T = json.loads(Path("checkpoints/R15_calibration.json").read_text())["temperature"]
ann_in = raw_logits(ann, make_loader(PlantDataset(manifest, "val_wild", CACHE_DIR, crops, eval_transform()), 128,
                                     train=False, num_workers=0), device)
ann_out = raw_logits(ann, make_loader(PathDataset(ood_rows.path, eval_transform()), 128, train=False, num_workers=0), device)
gates = {"R15 energy (deployed)": (familiarity_scores(ann_in, crop_idx, crop_mask, T)["energy"],
                                   familiarity_scores(ann_out.repeat(len(crops), 1), ood_crop, crop_mask, T)["energy"])}

# 2. CLIP as a gate: "is this a leaf of the crop the officer chose?"
#    the alternatives are fixed in advance and do NOT name the out-of-scope crops
GENERIC = ["a photo of a leaf of another plant.", "a photo of a fruit.", "a photo of a vegetable.",
           "a photo of a flower.", "a photo of soil.", "a photo of a person.", "a drawing or a page of text."]
scope_prompts = [f"a photo of a {CROP_TEXT[c][0]} leaf." for c in crops] + GENERIC
for name, clip in clip_models.items():
    with torch.no_grad():
        txt = clip["model"].encode_text(clip["tokenizer"](scope_prompts).to(device)).float()
    txt = (txt / txt.norm(dim=-1, keepdim=True)).cpu()
    scale = clip["model"].logit_scale.detach().exp().item()
    img_in = torch.load(f"features/clip_{name}_val_wild.pt")
    img_out = image_features(clip, ood_rows.path, device)
    p_in = (scale * img_in @ txt.T).softmax(dim=1)[torch.arange(len(img_in)), crop_idx]
    p_out = (scale * img_out.repeat(len(crops), 1) @ txt.T).softmax(dim=1)[torch.arange(len(ood_crop)), ood_crop]
    gates[f"{name} gate"] = (p_in, p_out)

rows = []
for gate, (s_in, s_out) in gates.items():
    sep = separation(s_in, s_out)
    flagged = np.asarray(s_out) < sep["threshold"]
    rows.append({"warning": gate, "AUROC": round(sep["auroc"], 3), "in-scope kept": f"{sep['in_scope_kept']:.0%}",
                 "wrong crop: missed": f"{1 - flagged[~same_crop].mean():.0%}",
                 "right crop, unknown problem: missed": f"{1 - flagged[same_crop].mean():.0%}"})
scope = pd.DataFrame(rows)
scope.to_csv("results/q6_scope_gate_val.csv", index=False)
print(scope.to_string(index=False))

out-of-scope: 695 photos x 6 crops = 4170 (photo, crop) pairs; 2 pairs are the right crop with a problem the model does not know
              warning  AUROC in-scope kept wrong crop: missed right crop, unknown problem: missed
R15 energy (deployed)  0.672           95%                88%                                100%
        OpenCLIP gate  0.910           95%                37%                                100%
         BioCLIP gate  0.864           95%                54%                                100%


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

clip_val = pd.read_csv("results/q6_clip_validation.csv").set_index("model")
# (model, lab F1-16, wild F1-16, family); ANN and baseline numbers from results/q3a_comparison_validation.csv
models = [
    ("random within crop",        0.265, 0.285, "reference"),
    ("C04 classical",             0.914, 0.322, "reference"),
    ("R01 initial ANN",           0.923, 0.359, "ANN"),
    ("final ANN (3 seeds)",       0.957, 0.471, "ANN"),
    ("OpenCLIP zero-shot",        clip_val.loc["OpenCLIP zero-shot (ensemble)", "val_lab F1-16"],
                                  clip_val.loc["OpenCLIP zero-shot (ensemble)", "val_wild F1-16"], "CLIP"),
    ("BioCLIP zero-shot",         clip_val.loc["BioCLIP zero-shot (ensemble)", "val_lab F1-16"],
                                  clip_val.loc["BioCLIP zero-shot (ensemble)", "val_wild F1-16"], "CLIP"),
]
COLOUR = {"reference": "#8a8984", "ANN": "#2a78d6", "CLIP": "#eb6834"}
STYLE = {"random within crop": ":", "C04 classical": "--", "R01 initial ANN": "--", "BioCLIP zero-shot": "--"}
SHORT = {"random within crop": "random", "C04 classical": "C04", "R01 initial ANN": "R01",
         "final ANN (3 seeds)": "ANN", "OpenCLIP zero-shot": "OpenCLIP", "BioCLIP zero-shot": "BioCLIP"}

def spread(values, gap=0.032):
    """Label heights: the values, pushed apart just enough that no two labels overlap."""
    order = sorted(range(len(values)), key=lambda i: values[i])
    out = list(values)
    for a, b in zip(order, order[1:]):
        out[b] = max(out[b], out[a] + gap)
    return out

fig, ax = plt.subplots(figsize=(7.2, 5.2))
for name, lab, wild, family in models:
    ax.plot([0, 1], [lab, wild], color=COLOUR[family], lw=2, ls=STYLE.get(name, "-"), marker="o", ms=8,
            markeredgecolor="white", markeredgewidth=2, zorder=3)
for side, x, ha in [(1, -0.06, "right"), (2, 1.06, "left")]:
    ys = spread([m[side] for m in models])
    for (name, lab, wild, family), y in zip(models, ys):
        text = f"{name}  {wild:.3f}" if side == 2 else f"{SHORT[name]}  {lab:.3f}"
        ax.text(x, y, text, ha=ha, va="center", fontsize=9, color="#0b0b0b")
ax.set_xticks([0, 1], ["lab photos\n(val_lab, PlantVillage)", "field photos\n(val_wild, PlantDoc)"])
ax.set_xlim(-0.5, 1.9)
ax.set_ylim(0.2, 1.02)
ax.set_ylabel("macro-F1 over the 16 disease classes")
for side in ("top", "right", "bottom"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#ebeae5", lw=0.8)
ax.set_axisbelow(True)
handles = [plt.Line2D([], [], color=COLOUR[f], lw=2, marker="o", ms=7, label=l)
           for f, l in [("ANN", "trained on lab photos (ANN)"), ("CLIP", "zero-shot, web-pretrained (CLIP)"),
                        ("reference", "references")]]
ax.legend(handles=handles, frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.45, -0.13), ncol=3)
ax.set_title("Models trained on lab photos lose half or more of their score on field photos;\nzero-shot CLIP, pretrained on web photos, does slightly better there",
             fontsize=10, loc="left")
fig.tight_layout()
fig.savefig("figures/q6_lab_vs_wild.png", dpi=200, bbox_inches="tight")

NameError: name 'pd' is not defined